# SleepFM — embeddings OSF PhysioNet 2018 (comparaison exploratoire, GPU)

Fichier : `notebooks/sleepfm_osf_physionet2018.ipynb`.

Option A du protocole de comparaison (`docs/OSF_COMPARISON.md`) : embeddings figés
du modèle OSF préentraîné sur les **500 sujets de développement** du découpage figé,
avec **les mêmes fenêtres que SleepFM**. Seuls 8 des 12 canaux OSF existent dans
PhysioNet 2018 (les 4 autres sont mis à zéro) : résultat **exploratoire**, biaisé
contre OSF. Aucune sonde ni score ici ; la comparaison se fait ensuite sur le Mac.
Les 494 sujets réservés ne sont pas utilisés.

**Sur Kaggle** :
1. `Settings > Accelerator > GPU T4 x2` et `Settings > Internet > On`.
2. Essai court : `LIMIT = 2`, puis `Run All`.
3. Run complet : `LIMIT = None`, puis `Save Version > Save & Run All (Commit)`.
4. Télécharger depuis l'onglet `Output` : `embeddings_physionet2018_osf.zip`,
   `osf_log.txt` et `pip_freeze.txt`.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Pas de GPU : Settings > Accelerator"
!df -h . | tail -1


## 1. Code, dépendances et poids OSF (Hugging Face, révision figée)


In [ ]:
import os
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
%cd {WORK}
REVISION = "origin/main"  # ou le SHA exact du commit, noté dans les sorties
![ -d sleepfm-pipeline ] || git clone -q https://github.com/sumarc183-design/sleepfm-pipeline.git
!cd sleepfm-pipeline && git fetch -q && git reset -q --hard {REVISION} && git log --oneline -1
%cd {WORK}/sleepfm-pipeline
!pip install -q -r requirements.txt
OSF_REVISION = "f2067f2d95b3bff43e4a690a471ba3d4457f12a7"
OSF_SHA256 = "c51190b1942556969af3c3d63c2e59430ddb1ea0377c50ea87df83712fc31857"
!mkdir -p third_party/osf
![ -f third_party/osf/osf_backbone.pth ] || curl -sL -o third_party/osf/osf_backbone.pth https://huggingface.co/yang-ai-lab/OSF-Base/resolve/{OSF_REVISION}/osf_backbone.pth
import hashlib
digest = hashlib.sha256(open("third_party/osf/osf_backbone.pth", "rb").read()).hexdigest()
if digest != OSF_SHA256:
    raise RuntimeError(f"Poids OSF inattendus : {digest}")
print("Poids OSF vérifiés")


## 2. Paramètres

- `LIMIT` : `2` pour l'essai court, `None` pour les 500 sujets.


In [ ]:
LIMIT = 2  # None pour le run complet (500 sujets)
OUT = f"{WORK}/embeddings/physionet2018_osf"


## 3. Embeddings OSF

Le lot suivant se télécharge pendant l'extraction et le calcul du lot en cours.
Relancer reprend où le run s'est arrêté.


In [ ]:
import subprocess
command = [
    "python", "-u", "scripts/embed_osf_physionet2018.py",
    "--split-file", "configs/split_physionet2018.json",
    "--osf-checkpoint", "third_party/osf/osf_backbone.pth",
    "--out", OUT,
    "--chunk-size", "8",
    "--batch-size", "256",
    "--num-threads", "2",
    *(["--limit", str(LIMIT)] if LIMIT else []),
]
with open(f"{WORK}/osf_log.txt", "w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
        log.write(line)
    return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"embed_osf_physionet2018.py a échoué (code {return_code}) : voir la sortie ci-dessus")


## 4. Garder le résultat

Sur le Mac, dézipper dans `embeddings/physionet2018_osf/`. La comparaison
(mêmes sondes, mêmes nuits, critère fixé avant) se prépare ensuite.


In [ ]:
import json, shutil
status = json.load(open(f"{OUT}/run_status.json"))
print(f"{len(status['embedded'])} sujet(s) faits, {len(status['rejected'])} rejeté(s), manquants : {status['missing'][:5]}")
print(f"Canaux plats mis à zéro : {status['flat_channels']}")
if not status["complete"]:
    raise RuntimeError("Run incomplet : relancer pour reprendre")
!python -m pip freeze > {WORK}/pip_freeze.txt
shutil.make_archive(f"{WORK}/embeddings_physionet2018_osf", "zip", OUT)
!ls -lh {WORK}/*.zip {WORK}/*.txt
